# Project 1 Module 5: QA/QC Practice

This notebook builds practical data-quality skills around `src/qa.py`. You will inspect deterministic sandbox layers, read `QAResult` values, write reports only inside temporary directories, and assemble a quality gate without touching project outputs.

The goal is to practise predicting, testing, and debugging QA/QC behavior rather than copying the production orchestrator.

## How to Use This Notebook

1. Predict each result before running code.
2. Complete only the cell marked for the current exercise.
3. Keep every file operation inside `TemporaryDirectory`.
4. Read `result.passed`, the individual counts, and `result.error` before changing code.
5. Make one defect at a time so each observation has a clear cause.
6. Open the optional solutions only after a genuine attempt.

**Safety rule:** import `run_qa` and `REPORT_PATH` for API recognition only. Do not call `run_qa()` here because it targets configured project layers. Never pass `REPORT_PATH` to `write_report` in this notebook.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import csv
import sys

import geopandas as gpd
from shapely.geometry import LineString, Point, Polygon

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "practice":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.qa import (
    REPORT_PATH,
    QAResult,
    QualityGateError,
    inspect_layer,
    run_qa,
    write_report,
 )

EXPECTED_CRS = "EPSG:3347"
REQUIRED_FIELDS = ["feature_id", "name"]


def passed(exercise: str) -> None:
    print(f"PASS: {exercise}")


def write_sample(path: Path, ids=(1, 2), crs=EXPECTED_CRS) -> Path:
    frame = gpd.GeoDataFrame(
        {
            "feature_id": list(ids),
            "name": [f"site-{index}" for index in range(len(ids))],
        },
        geometry=[Point(index, index) for index in range(len(ids))],
        crs=crs,
    )
    frame.to_file(path, driver="GeoJSON")
    return path


print("Safe QA practice environment ready.")
print(f"Production report path (do not use here): {REPORT_PATH}")

# Part 1: QA Concepts and a Passing Baseline

A **blocking check** prevents delivery when it fails. In `inspect_layer`, empty data, missing schema, geometry defects, ID defects, CRS mismatch, and actionable read errors all contribute to `passed=False`.

An **informational check** describes context without deciding the gate by itself. `row_count`, `checked_at_utc`, and `input_path` help explain what was inspected, although an empty `row_count` is also used by the blocking empty-dataset rule.

## Exercise 1A

Classify the listed observations as blocking or informational. Keep the two lists disjoint.

In [ ]:
observations = [
    "duplicate_id_count",
    "checked_at_utc",
    "invalid_geometry_count",
    "input_path",
    "crs_mismatch",
]

# TODO: distribute every observation into exactly one list.
blocking_checks = []
informational_checks = []

In [ ]:
# Completed smoke check: this cell creates no persistent files.
with TemporaryDirectory() as temporary:
    smoke_path = write_sample(Path(temporary) / "valid_sites.geojson")
    smoke_result = inspect_layer(
        dataset_name="valid_sites",
        input_path=smoke_path,
        expected_fields=REQUIRED_FIELDS,
        id_field="feature_id",
        expected_crs=EXPECTED_CRS,
    )

assert isinstance(smoke_result, QAResult)
assert smoke_result.passed is True, smoke_result
assert smoke_result.row_count == 2
assert smoke_result.error == ""
passed("Passing baseline smoke check")

# Part 2: Reading a Dataclass Result

`QAResult` is a frozen dataclass. Read its named attributes rather than relying on position. A useful diagnosis starts with `passed` and `error`, then examines the individual counts and CRS values.

## Exercise 2A

Read `smoke_result` from the completed baseline. Fill the variables with attributes from the result, then explain why the layer passed.

In [ ]:
# TODO: replace None with QAResult attribute reads.
baseline_dataset = None
baseline_rows = None
baseline_passed = None
baseline_error = None
baseline_explanation = ""

print(
    baseline_dataset,
    baseline_rows,
    baseline_passed,
    baseline_error,
    baseline_explanation,
 )

In [ ]:
# Run after completing Exercise 2A.
assert baseline_dataset == "valid_sites"
assert baseline_rows == 2
assert baseline_passed is True
assert baseline_error == ""
assert baseline_explanation.strip(), "Explain the evidence, not only the Boolean."
passed("Exercise 2A")

# Part 3: Required Schema and Empty Data

`expected_fields` lists non-geometry fields that must be present. `inspect_layer` also checks the active geometry column. A missing required field increments `missing_field_count`; a missing ID field also gives an actionable `error`.

## Exercise 3A: Required Schema

Create a sandbox layer without `name`, inspect it with `REQUIRED_FIELDS`, and predict both `missing_field_count` and `passed`.

In [ ]:
with TemporaryDirectory() as temporary:
    schema_path = Path(temporary) / "missing_name.geojson"
    schema_frame = gpd.GeoDataFrame(
        {"feature_id": [1, 2]},
        geometry=[Point(0, 0), Point(1, 1)],
        crs=EXPECTED_CRS,
    )
    schema_frame.to_file(schema_path, driver="GeoJSON")

    # TODO: call inspect_layer with REQUIRED_FIELDS.
    schema_result = None

predicted_missing_fields = None  # TODO: replace with an integer.
predicted_schema_passed = None  # TODO: replace with a Boolean.

## Exercise 3B: Empty Dataset

A layer can have the right columns and CRS but still be unusable because it has zero rows. Build an empty GeoDataFrame with `feature_id`, `name`, and geometry; write it to a temporary GeoJSON if your GeoPandas/GDAL version supports that operation.

If writing an empty GeoJSON is unsupported in your environment, capture the exception text and explain why a zero-row fixture may need a different test format or a mocked read boundary.

In [ ]:
empty_result = None
empty_write_error = ""

with TemporaryDirectory() as temporary:
    empty_path = Path(temporary) / "empty_sites.geojson"
    empty_frame = gpd.GeoDataFrame(
        {"feature_id": [], "name": []},
        geometry=[],
        crs=EXPECTED_CRS,
    )
    try:
        # TODO: write empty_frame, then inspect it.
        pass
    except (OSError, ValueError) as exc:
        empty_write_error = str(exc)

print(empty_result if empty_result is not None else empty_write_error)

# Part 4: Geometry Defects

`inspect_layer` counts three distinct geometry problems:

- **Null:** no geometry value is present.
- **Empty:** a geometry object exists but contains no coordinates.
- **Invalid:** coordinates exist, but the geometry violates validity rules, such as a self-intersecting polygon.

## Exercise 4A

Create one fixture for each defect. Keep IDs unique so the geometry check is isolated. Inspect each temporary GeoJSON and record the count that should increase.

In [ ]:
null_geometry = None
empty_geometry = Point()
invalid_geometry = Polygon([(0, 0), (2, 2), (0, 2), (2, 0), (0, 0)])

geometry_cases = {
    "null_geometry": null_geometry,
    "empty_geometry": empty_geometry,
    "invalid_geometry": invalid_geometry,
}
geometry_results = {}

with TemporaryDirectory() as temporary:
    for case_name, geometry in geometry_cases.items():
        case_path = Path(temporary) / f"{case_name}.geojson"
        case_frame = gpd.GeoDataFrame(
            {"feature_id": [1], "name": [case_name]},
            geometry=[geometry],
            crs=EXPECTED_CRS,
        )
        # TODO: write case_frame, inspect it, and store the QAResult.
        geometry_results[case_name] = None

# TODO: replace None with the relevant QAResult count attribute name.
geometry_count_attributes = {
    "null_geometry": None,
    "empty_geometry": None,
    "invalid_geometry": None,
}

# Part 5: ID Nulls and Duplicates

When `id_field` exists, null IDs and repeated non-null IDs are blocking. The duplicate count is the number of repeated occurrences after the first, not the number of distinct duplicated values.

## Exercise 5A

Inspect `[1, None, 2]` and `[1, 1, 2, 2]` in separate temporary layers. Predict the two relevant counts before completing the calls.

In [ ]:
id_results = {}
id_cases = {
    "null_id": [1, None, 2],
    "duplicate_ids": [1, 1, 2, 2],
}

with TemporaryDirectory() as temporary:
    for case_name, ids in id_cases.items():
        case_path = write_sample(Path(temporary) / f"{case_name}.geojson", ids=ids)
        # TODO: inspect case_path and store the result by case_name.
        id_results[case_name] = None

predicted_null_id_count = None  # TODO
predicted_duplicate_id_count = None  # TODO

# Part 6: CRS Checks

CRS validation compares the string recorded from the layer with `expected_crs`. A valid geometry in the wrong coordinate reference system still fails the gate.

## Exercise 6A

Write a layer in `EPSG:4326`, then inspect it while expecting `EPSG:3347`. Read both CRS fields from the result and explain why reprojection belongs before QA rather than inside `inspect_layer`.

In [ ]:
with TemporaryDirectory() as temporary:
    wrong_crs_path = write_sample(
        Path(temporary) / "wrong_crs.geojson",
        crs="EPSG:4326",
    )
    # TODO: inspect while expecting EXPECTED_CRS.
    crs_result = None

observed_crs = None  # TODO: read from crs_result.
required_crs = None  # TODO: read from crs_result.
crs_explanation = ""  # TODO: explain the failed responsibility boundary.

# Part 7: Interpreting `passed` and `error`

`passed=False` does not always imply a non-empty `error`. Counted quality defects normally leave `error` empty, while missing files and missing ID fields provide an error message. Diagnose both channels.

## Exercise 7A

Complete `summarize_result` so it distinguishes read/schema errors, counted quality failures, and clean passes.

In [ ]:
def summarize_result(result: QAResult) -> str:
    # TODO: return "error: ...", "quality failure", or "passed".
    return "TODO"


missing_file_result = inspect_layer(
    dataset_name="missing_sandbox",
    input_path=Path("definitely-not-created-sandbox.geojson"),
    expected_fields=REQUIRED_FIELDS,
    id_field="feature_id",
    expected_crs=EXPECTED_CRS,
 )

missing_file_summary = summarize_result(missing_file_result)
baseline_summary = summarize_result(smoke_result)
print(missing_file_summary, baseline_summary, sep="\n")

# Part 8: Writing a CSV Report Safely

`write_report` serializes every dataclass field in a stable column order and creates parent directories as needed. In this workbook, the destination must be beneath a live `TemporaryDirectory`.

## Exercise 8A

Write a report containing the passing baseline and missing-file result. Read it back with `csv.DictReader`, then verify that Boolean values are represented as CSV text.

Do not use the default argument and do not pass `REPORT_PATH`.

In [ ]:
report_rows = []

with TemporaryDirectory() as temporary:
    sandbox_report = Path(temporary) / "qa" / "practice_report.csv"
    # TODO: write [smoke_result, missing_file_result] to sandbox_report.
    # TODO: open sandbox_report with newline="" and encoding="utf-8".
    # TODO: assign list(csv.DictReader(file)) to report_rows.
    pass

print(report_rows)

In [ ]:
# Run after completing Exercise 8A.
assert len(report_rows) == 2
assert report_rows[0]["dataset"] == "valid_sites"
assert report_rows[0]["passed"] == "True"
assert report_rows[1]["passed"] == "False"
assert "Missing processed file" in report_rows[1]["error"]
passed("Exercise 8A")

# Part 9: Controlled Defect Injection

A strong QA experiment starts from one known-good fixture, changes exactly one property, and predicts the affected `QAResult` fields. This prevents a second defect from obscuring the first.

## Exercise 9A

Complete `evaluate_fixture` and use it to compare a baseline, duplicate-ID layer, wrong-CRS layer, and missing-field layer. Return results; do not print inside the helper.

In [ ]:
def evaluate_fixture(dataset_name: str, path: Path) -> QAResult:
    # TODO: delegate to inspect_layer with the shared schema and CRS.
    raise NotImplementedError("Complete evaluate_fixture")


def expected_signal(result: QAResult) -> str:
    # TODO: return the name of the one deliberately triggered defect.
    return "TODO"


controlled_results = {}
# TODO: create each fixture inside one TemporaryDirectory, evaluate it,
# and fill controlled_results without writing anywhere else.